# 18.10 蒸餾是否幫到學生？


蒸餾學生答對更多題，是否一定是教師訊號的功勞？如果它比基準多讀了一倍資料，或學生架構也變大，答案還不能歸因蒸餾。比較先固定[學生架構](https://birdhackor.github.io/tiny-perceptron-vlm/18.2.html)，把同一初始化、訓練資料與已讀token預算分成兩條路：普通SFT與加入蒸餾，最後在同一份獨立真值上評估。SFT是監督式微調，以理想助手回答教學生；加入蒸餾則另外讀教師的候選比例。這份獨立評估題沒有用來更新學生，答案真值也不由教師的猜測代填。

教師計算並不免費。假設兩種方法學生部分各花10秒，訓練100步、讀1000個token；蒸餾另花40秒教師forward（前向計算：把輸入交給教師，算出候選分數與比例），總時間就是50秒，不是10秒。按學生更新預算比較能回答「多給教師訊號有沒有幫忙」；按同樣總時間比較則回答另一個問題，蒸餾可能只能做較少更新。兩種公平應分開報。

下面的人工表沿用這個時間情境，兩個學生都設為6104個參數、100次更新、1000個訓練token。我們再假設四題中分別答對三題與四題，練習同時讀品質與成本；這些構造數字不是模型實測。

| 方法 | 答對數／四題 | 學生時間 | 教師時間 | 合計 |
| --- | ---: | ---: | ---: | ---: |
| SFT | 3/4（75%） | 10秒 | 0秒 | 10秒 |
| 蒸餾 | 4/4（100%） | 10秒 | 40秒 | 50秒 |

假設的蒸餾版答對更多，也多花教師時間。這張表只能說明報告應呈現什麼；要證實改善，必須保存真實訓練結果，再用足夠獨立題目與重複實驗檢查。重算正確率與總秒數的原Python放在本節最後。

部署又是另一筆成本。這裡兩個學生都是6104個參數，架構沒有因蒸餾而再縮小；教師與較小學生之間的節省，不能當成普通小學生與蒸餾小學生之間的節省。同架構的實際耗時仍會受回答長度與執行條件影響，應量測成品的延遲、吞吐與記憶體，不能由教師訊號或權重大小直接填出速度。較少學生更新、較少總訓練時間與較便宜部署，回答的是三個不同問題。

評價不能只用「與教師回答一致率」。教師若給錯答案，學生模仿得更像可能反而退步；應用真值判內容，用格式規格判結構，並重跑既有誠實與安全情境。同樣正確率下，也可能一組善於短題、另一組在長題失敗，所以應按題型或輸入條件拆開看。

若教師先生成離線文字，成本改成資料生成與驗證，蒸餾訓練時不一定每步教師forward；仍要另外列出這些一次性成本。正式白盒訓練需要可載入、已檢查目標任務能力的教師權重；能力不足可以作失敗對照，但要明說，不能把本章的隨機教師檢查當作可靠教師。需要保存版本與[詞表對齊](https://birdhackor.github.io/tiny-perceptron-vlm/18.6.html)條件，讓比較能重現。

練習換一個人工情境：普通與蒸餾小學生的準備時間仍是10秒與50秒，但部署時都只需每次1毫秒。能否說請求次數夠多，就會從推論節時收回多花的40秒？若改成比較每次10毫秒的教師與每次1毫秒的學生，又應先確認哪些品質與成本條件？請說出比較的兩個對象，以及你正在算準備成本還是使用成本。

<details>
<summary>核對你的推理</summary>

前一對小學生每次耗時相同，沒有推論節時可累積；蒸餾若提高品質，那是另一項收益。後一對的每次成本才有差距，仍須確認學生品質可接受、計時設備與輸入條件可比，並加計準備教師訊號與學生所需成本。預先快取只改變成本發生的時點，沒有把生成快取變成免費。

</details>

<details>
<summary>選讀：共同評估指標與既有結果入口</summary>

回答NLL是negative log-likelihood（負對數似然）：在標準回答前文上，把模型給下一個真值token的機率取負自然對數，越低表示標準答案越容易預測。將全部有效回答與EOS位置的代價加總，再除以有效位置總數，就得到共同真值下的平均代價。它沒有加入教師KL，因此不同訓練目標的學生仍可用同一套評估比較；不能直接拿各自的混合訓練loss排名。

正式屬性對照在同一寬度下固定初始化、訓練題目與有效目標預算，比較普通真值訓練、教師硬回答與加入教師分布三條路。完整條件、各寬度結果、教師成本及逐題輸出保存在[蒸餾實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/distillation.json)。這是單一seed的小型合成任務，不能由一次持平或退步推論蒸餾普遍無效。

域外診斷另外受到上下文預算限制：只計題目、聊天提示、生成空間與標準答案都能完整放入的題目，沒有截短題目再配完整答案。這樣的少量診斷不能稱為完整GSM8K benchmark；選題條件與輸出也保存在上述實報。

</details>

<details>
<summary>選讀：用Python重算人工表的正確率與總時間</summary>


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
runs = [
    {
        "method": "SFT",
        "student_params": 6104,
        "steps": 100,
        "tokens": 1000,
        "student_seconds": 10,
        "teacher_seconds": 0,
        "correct": 3,
        "total": 4,
    },
    {
        "method": "distill",
        "student_params": 6104,
        "steps": 100,
        "tokens": 1000,
        "student_seconds": 10,
        "teacher_seconds": 40,
        "correct": 4,
        "total": 4,
    },
]
for run in runs:
    accuracy = run["correct"] / run["total"]
    seconds = run["student_seconds"] + run["teacher_seconds"]
    print(
        run["method"],
        "參數",
        run["student_params"],
        "步數",
        run["steps"],
        "token",
        run["tokens"],
        "正確率",
        accuracy,
        "總秒數",
        seconds,
    )

輸入字典已經有人工指定的條件與結果，程式只計算衍生指標。SFT輸出6104參數、100步、1000token、0.75、10秒；蒸餾同樣學生預算、1.0、50秒。字典中的`teacher_seconds`是教師時間，不是學生更新的一部分。

</details>
